In [ ]:
# Solutions live in solutions/. Run from the repository root so imports and paths work.
import os, pathlib, sys
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "workshop").is_dir())
os.chdir(_root)
sys.path.insert(0, str(_root))

# Module 4: Create a custom widget with AI help

**Time:** 35 minutes

1. When no existing widget fits
2. The smallest anywidget: one class, one JavaScript string
3. Review an AI-drafted widget with a checklist (it has bugs!)
4. Use the reviewed widget for real analysis
5. Your turn: draft your own widget with an AI assistant

You will **read** some JavaScript, but you do not need to write it from scratch.

In [ ]:
# Live reload: when you save a widget's .js or .css file, open widgets update.
# This must run before anywidget is imported.
%env ANYWIDGET_HMR=1

## 1. When no existing widget fits (2 minutes)

**The task:** you recorded a long signal (EEG, ECG, a sensor, a microphone) and
need to **mark time ranges** with labels such as "artifact" or "seizure", then
use those ranges in pandas. None of the widgets in Module 3 does this. Time to
build one.

In [ ]:
import numpy as np

# A synthetic 10-minute recording at 250 Hz (150,000 samples), similar to EEG:
# a 10 Hz rhythm, slow drift, noise, and four bursts of movement artifact.
rng = np.random.default_rng(42)
fs = 250
t = np.arange(0, 600, 1 / fs)
y = 0.5 * np.sin(2 * np.pi * 10 * t) + 0.3 * np.sin(2 * np.pi * 0.2 * t) + rng.normal(0, 0.3, t.size)
for start in [42, 130.5, 301, 455]:
    burst = (t >= start) & (t < start + 2.5)
    y[burst] += rng.normal(0, 4, burst.sum())
print(f"{t.size:,} samples, {t[-1]:.0f} seconds")

## 2. The smallest anywidget (8 minutes)

An anywidget is a Python class with:

- **traits** tagged `sync=True`: the shared state (same as Module 1)
- **`_esm`**: a JavaScript module with a `render` function that draws the widget

In `render`, the JavaScript uses four calls:

| JavaScript | Meaning |
|---|---|
| `model.get("count")` | read a synced trait |
| `model.set("count", 5)` | change it in the browser |
| `model.save_changes()` | send the change to Python |
| `model.on("change:count", fn)` | run `fn` when it changes (from either side) |

In [ ]:
import anywidget
import traitlets


class Counter(anywidget.AnyWidget):
    _esm = """
    export default {
      render({ model, el, signal }) {
        const button = document.createElement("button");
        const update = () => { button.textContent = `Count: ${model.get("count")}`; };
        update();

        button.addEventListener("click", () => {
          model.set("count", model.get("count") + 1);
          model.save_changes();
        }, { signal });

        model.on("change:count", update);
        signal.addEventListener("abort", () => model.off("change:count", update));
        el.append(button);
      }
    };
    """
    count = traitlets.Int(0).tag(sync=True)


counter = Counter()
counter

Click the button a few times. Then run these cells:

In [ ]:
counter.count

In [ ]:
counter.count = 100  # Python -> browser
display(counter)     # a second view of the same model

**Why `signal`?** When a view is removed (you clear or re-run the cell), the
browser "aborts" its `signal`. Listeners registered with it are removed, so old
views stop listening. Forgetting this is the most common widget bug.

For real widgets, the JavaScript lives in its own file. Look at the reviewed
widget used in this module:

In [ ]:
import pathlib

folder = pathlib.Path("workshop/range_labeler")
for path in sorted(folder.rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts:
        print(f"{str(path):45s} {len(path.read_text().splitlines()):4d} lines")

## 3. Review an AI-drafted widget (12 minutes)

We asked an AI assistant:

> *"Make an anywidget that plots a time series and lets me drag to mark
> labelled time ranges. I want the ranges back in pandas."*

Its answer is in `ai/draft_widget/`. It looks reasonable, and it runs. Try it:
pick a label, then drag across the plot.

(We give the draft a shorter signal: the first 60 seconds.)

In [ ]:
from ai.draft_widget import RangeLabelerDraft

first_minute = t < 60
draft = RangeLabelerDraft(y[first_minute], t=t[first_minute], labels=["artifact", "event"])
draft

In [ ]:
draft.ranges_df

Nothing appears when you drag, and Python gets nothing. **Your task:**
open [`ai/review_checklist.md`](ai/review_checklist.md) and the two draft files:

- [`ai/draft_widget/__init__.py`](ai/draft_widget/__init__.py)
- [`ai/draft_widget/static/widget.js`](ai/draft_widget/static/widget.js)

Go through the checklist and find as many bugs as you can. There are **at
least seven**. Write them in the cell below.

<details>
<summary><b>Hint 1</b> (click to open)</summary>

Look at what happens to the `ranges` array in the `mouseup` handler (checklist A2),
and what the "Clear all" button does *not* do (A1).
</details>

<details>
<summary><b>Hint 2</b></summary>

Compare how the chips are built (B4), what happens to `model.on` when the view
is removed (B2), and how `t` and `y` are declared in Python (C1).
</details>

<details>
<summary><b>Hint 3</b></summary>

Try `RangeLabelerDraft([1, 2, 3], t=[0, 1])` (D1). Look at how `_esm` is set (E1).
</details>

In [ ]:
my_findings = """
1. A1  "Clear all" calls model.set without model.save_changes(): Python keeps the old ranges.
2. A2  mouseup pushes into the array from model.get and sets the same array: no change event,
       so nothing is drawn and Python receives nothing.
3. B2  model.on("change:ranges") is never removed: old views keep listening after a cell is cleared.
4. B4  chips use innerHTML with the label: a label like <img src=x onerror=alert(1)> runs as HTML.
5. C1  t and y are traitlets.List: 1 million samples = 30 MB of JSON instead of 12 MB binary.
6. D1  no validation: RangeLabelerDraft([1, 2, 3], t=[0, 1]) is accepted silently.
7. E1  _esm and _css are relative strings: they only work when the kernel runs in the repo root.
Bonus: Math.min(...y) crashes for ~300k+ samples; fixed 800 px canvas; colours ignore dark mode.
"""
print(my_findings)

## 4. The reviewed widget, used for real analysis (5 minutes)

`workshop/range_labeler/` is the fixed version: every checklist item passes,
and it draws a **million samples** smoothly (min/max per pixel column). Here it
gets the full 10-minute recording.

Mark the four noisy bursts as **artifact** (zoom with the wheel, Shift-drag to
pan, double-click to reset).

In [ ]:
from workshop.range_labeler import RangeLabeler

labeler = RangeLabeler(y, t=t, labels=["artifact", "event"])
labeler

In [ ]:
labeler.ranges_df

Marking by hand is slow for long recordings. A common workflow: let a simple
detector **propose** ranges from Python, then **correct** them by hand in the
plot (drag the edges again, or remove a chip with ×). Here the detector flags
1-second windows with a high standard deviation:

In [ ]:
import pandas as pd

rolling_std = pd.Series(y).rolling(fs, center=True).std().to_numpy()
noisy = (rolling_std > 1.5).astype(int)
edges = np.flatnonzero(np.diff(noisy))
for start, end in zip(edges[::2], edges[1::2]):
    labeler.add_range(t[start], t[end], "artifact")
labeler.ranges_df

Now use the labels: compare the signal with and without the artifacts.

In [ ]:
artifact = labeler.mask("artifact")
print(f"marked as artifact: {artifact.mean():.1%} of samples")
print(f"standard deviation, all samples:       {y.std():.3f}")
print(f"standard deviation, artifacts removed: {y[~artifact].std():.3f}")

## 5. Your turn: draft a widget with AI (10 minutes)

1. Pick an idea from [`ai/task_ideas.md`](ai/task_ideas.md), or use your own.
2. Create a starter folder (run the cell below with your own name).
3. Open [`ai/prompt_template.md`](ai/prompt_template.md), fill in the task,
   and paste it into your AI assistant. **No assistant?** Edit the starter
   files by hand, or study `workshop/range_labeler/`.
4. Replace the three starter files with the assistant's code. Run the cell
   that displays the widget again.
5. Review the result with the checklist.

In [ ]:
from workshop.scaffold import new_widget

folder = new_widget("my_first_widget", overwrite=True)
print("Your widget folder:", folder)

In [ ]:
import importlib

import my_widgets.my_first_widget as mine  # change the name if you changed it

importlib.reload(mine)  # pick up changes to __init__.py
widget = mine.MyFirstWidget()
widget

**Tips**

- With `ANYWIDGET_HMR=1` (first cell), saving `widget.js` or `widget.css`
  updates the widget at once. Changes to `__init__.py` need the reload cell above.
- Nothing shows? Open the browser console (F12). Paste the error to your assistant.
- Before you trust the code, run the checklist. Ask the assistant to fix one
  checklist item at a time.

**Next:** [Module 5, package and publish](05_package_and_publish.ipynb)